In [ ]:
from itertools import zip_longest as zip 
from itertools import chain
from itertools import product

"""
A_q(lambda) of Unitary group. 

The weight lattice is Z^n for U(p,q) with n = p+q. 
We will use half integral weights. 
In the following tuple of integer represents the 2 * weight. 
"""

def tworho(n):
    return list(n - 2*i - 1for i in range(n))

def smul_list(s,a):
    return [s * x for x in a]

def sum_list(a, b):
    """
    The sum of tuples a and b is given by
    c[i] = a[i] + b[i]
    """
    a,b = list(a),list(b)
    assert len(a) == len(b), "Tuples must have the same length"
    return [x + y for x, y in zip(a, b)]


def sub_list(a, b):
    """
    The sum of tuples a and b is given by
    c[i] = a[i] - b[i]
    """
    a,b = list(a),list(b)
    assert len(a) == len(b), "Tuples must have the same length"
    return [x - y for x, y in zip(a, b)]

"""
The A_q(lambda) is given by list of tuples D:=[(p_i,q_i,l_i)] where l_i denote the character of $U(p_i,q_i)\to U(1)$.  
In particular the theta-stable q is defined by Q := [(p_i,q_i)]
"""

"""
The list [(p_i,q_i)] give a compact L if and only if p_i q_i = 0 for all i. 
This is the case when A_q(lambda) can be a discrete series. 
"""
def is_Lq_discrete(Q): 
    return all(p*q==0 for p,q in Q)


def sizeQ(Q):
    return sum(p+q for p,q in Q)

def signQ(Q):
    """
    return the signature of U(p,q) 
    """
    return sum(p for p,_ in Q), sum(q for _,q in Q)

def tworhou(Q): 
    """
    The function compute rho(u)
    We use rho(u) = rho - rho_L where rho_L is the rho of the Levi
    """
    return sub_list(tworho(sizeQ(Q)),chain(*[tworho(p+q) for p,q in Q]))

def QtoSign(Q):
    """
    From {(p_i,q_i)} generate the sign string e_1 e_2 ... e_n with + ... + (p_i) - ...- (q_i)
    """
    res = []
    for pi, qi in Q:
        res += [1]*pi + [-1]*qi
    return res

def PM2Sign(s):
    """
    From +- sequence to sign 
    """
    return [1 if e == '+' else -1 for e in s] 

def Sign2PM(s):
    return ''.join('+' if e == 1 else '-' for e in s)


def tworhok(Q):
    """
    Note that 2 * rho_K is always of the form (p-1,..., -p+1) (q-1,...-q+1) where the number are distributed on 
    compact groups
    """
    p, q = signQ(Q)
    rhop, rhoq = tworho(p), tworho(q)
    res = []
    for pi,qi in Q:
        res = res + rhop[:pi]+rhoq[:qi]
        rhop, rhoq = rhop[pi:], rhoq[qi:]
    return res

def arrangebysign(s,t1,t2):
    """ 
    Suppose s has (p,q) +1 and -1, then len(t1) = p, len(t2) = q.
    The result is the string by put t1 at +1 sign and t2 at -1 sign. 
    """
    p , q = s.count(+1), s.count(-1)
    assert(p == len(t1) and q == len(t2))
    res = []
    for e in s :
        if e == 1 :
            res.append(t1[0])
            t1 = t1[1:]
        else:
            res.append(t2[0])
            t2 = t2[1:]
    return res

def pickbysign(s,t):
    """ 
    Suppose s has (p,q) +1 and -1, then len(t) = len(s).
    The result is the a pair by put elements in t with +1 sign -1 sign. 
    """
    assert(len(s)==len(t))
    t1 , t2 = [], []
    for e in s:
        if e == +1 : 
            t1.append(t[0])
        else:
            t2.append(t[0])
        t = t[1:]
    return (t1,t2)
  

def tworhok_1(Q):
    """
    Note that 2 * rho_K is always of the form (p-1,..., -p+1) (q-1,...-q+1) where the number are distributed on 
    compact groups
    """
    p, q = signQ(Q)
    rhop, rhoq = tworho(p), tworho(q)
    sign = QtoSign(Q)
    res = arrangebysign(sign,rhop,rhoq)
    return res

def tworholk(Q):
    """
    2 * rho_(l \cap k)
    """
    res = []
    for pi,qi in Q:
        res = res + tworho(pi)+tworho(qi)
    return res

def tworhouk(Q):
    """
    rho_(u \cap k) = rho_k - rho_{l\cap k}
    """
    return sub_list(tworhok(Q),tworholk(Q))

def tworhoup(Q):
    """
    rho_(u \cap p) = rho_u - rho_{u\cap k}
    """
    return sub_list(tworhou(Q),tworhouk(Q))


"""
A complete L-paramter of Unitary group or rank n is a pair of a list of numbers (a_1, ...,  a_n) and 
a list of sign (e_1, ..., e_n)
"""


"""
In the following assume A_q(lambda) is a discrete series. We 
The Harish-Chandra parameter of A_q(lambda) is nothing but lambda+rho with respect to a postive system compatible with q. 
The missing data is the choice of positive system. 
"""

def lamh(Q,lam):
    """
    Give a list of signature, lam a list of integers
    lamh = l_1 ... l_1 l_2 ... l_2 ... l_k ... l_k where l_i occure p_i+q_i times. 
    """
    assert(len(Q) == len(lam))
    res = []
    for (pi,qi),l in zip(Q,lam):
        res += [l]*(pi+qi)
    return res

def HCParameter(Q,lam):
    """
    Harisch-Chandra paramter as weight of U(p) x U(q)
    """
    n = sizeQ(Q)
    gamma = sum_list(lamh(Q,lam),tworho(n))
    t1,t2 = pickbysign(QtoSign(Q),gamma) 
    return (tuple(t1),tuple(t2))

def MinimalKtype(Q,lam=None):
    """
    It is given by lam + 2rho_up
    When A_q(lam) is a discrete series it is also called the Blattner paramter.
    """    
    if not lam:
        lam = [0]*len(Q)
    Lam = sum_list(lamh(Q,lam),smul_list(2,tworhoup(Q)))
    t1,t2 = pickbysign(QtoSign(Q),Lam) 
    return (tuple(t1),tuple(t2))


In [ ]:
"""
Give n generate all possible signs 
"""
def all_signs(n):
    return product([1,-1],repeat=n)

def sign2Q(s):
    """
    give a tuple of sign, return the Q for discrete series. 
    Break a tuple of +1 and -1 into (p,0) and (0,q) tuples.
    Args:
    input_tuple (tuple): A tuple containing +1 and -1 values.
    
    Returns:
    list: A list of tuples (p,0) and (0,q) representing consecutive +1's and -1's.
    """
    if len(s) == 0: return []
    res = []
    count = 0
    current = s[0] 

    for value in chain(s, [-1*s[-1]]):
        if value != current:
            res.append((count, 0) if current == 1 else (0, count))
            current = value
            count = 1
        else:
            count += 1

    return res



In [ ]:
"""
Signed Young diagram is give by a list [(l_i,p_i,q_i)] where l_i is the length and p_i, q_i are the signature. 
"""
def regSYD(SY):
    res = dict()
    for l,p,q in SY:
        p0,q0 = res.get(l,(0,0))
        res[l] = p0+p,q0+q
    res = [(l,p,q) for l,(p,q) in res.items() if p!=0 or q!=0]
    res = sorted(res,key=lambda x: x[0],reverse=True)
    return res

def pmstr(n,e):
    if e == 1:
        return '+-'*(n//2)+'+'*(n%2)
    else:
        return '-+'*(n//2)+'-'*(n%2)


def strSYD(SY):
    #SY = sorted(SY,key = lambda x: x[0], reverse=True)
    SY = regSYD(SY)
    SY = sorted(SY, reverse=True)
    res = ''
    for l,p,q in SY:
        res += '\n'.join([pmstr(l,1)]*p+[pmstr(l,-1)]*q)+'\n'
    return res

def addright(SY,p,q):
    """
    Add signs on the right of the diagram
    """
    res = []
    SY = regSYD(SY)
    for l,pi,qi in SY: 
        # in the following ri, si represent number 
        # of + and - sign adding each time
        if l%2 == 0 :
            # even row add (+,-)
            ri ,si = min(pi,p),min(qi,q)
            res.append((l+1,ri,si))
            res.append((l,pi-ri,qi-si))
        else:
            # odd row add (-,+)
            si ,ri = min(pi,q),min(qi,p)
            res.append((l+1,si,ri))
            res.append((l,pi-si,qi-ri))
        p -= ri
        q -= si
    res.append((1,p,q))
    return regSYD(res)

def addleft(SY,p,q):
    """
    Add signs on the left of the diagram
    """
    res = []
    SY = regSYD(SY)
    for l,pi,qi in SY: 
        # in the following ri, si represent number 
        # of + and - sign adding each time

        # add (-,+) 
        # note that the sign of the row should be change
        ri ,si = min(pi,q),min(qi,p)
        res.append((l+1,si,ri))
        res.append((l,pi-ri,qi-si))
        p -= si
        q -= ri
    res.append((1,p,q))
    return regSYD(res)

def AVofAq(Q):
    """
    The AV only depends on Q
    """
    SY = []
    for p,q in reversed(Q):
        SY = addright(SY,p,q) 
    return SY


In [ ]:
""" Test """
Q = [(2,0),(0,3),(4,0),(0,2),(2,0)]


print("rho_u", tworhou(Q))
print("rho_k", tworhok(Q))
print("rho_k", tworhok_1(Q))

print("rho_up",tworhoup(Q))



In [ ]:
print("rho_up",tworhoup([(2,0),(2,0),(0,2),(3,0)]))
print("rho_up",tworhoup([(4,0),(0,2),(3,0)]))

In [ ]:
n=5
SS = list(all_signs(n))
print(list(all_signs(n)))
BP = set() 
for s in SS:
    print('-'*20)
    Q = sign2Q(s)
    print(Sign2PM(s),Q)
    print(AVofAq(Q))
    print(strSYD(AVofAq(Q)))

    #print(sizeQ(Q),Q)
    b = MinimalKtype(Q)
    print(b)
    BP.add(b)

assert(len(BP) == 2**n)


In [ ]:
SY = [(3,2,1),(2,1,2),(5,3,0),(5,0,3),(5,1,2),(4,0,3)]
print(strSYD(SY))
